# Gemma 4 12B Monopoly QLoRA pilot

This is the primary staged notebook for the `ppo-plus-v2` pilot. It trains and gates a randomized-opponent PPO teacher, rollout-relabels compact canonical decisions, fine-tunes `unsloth/gemma-4-12b-it` with the proven 15 GB T4 profile, and evaluates Gemma as the standalone player.

The laptop launcher writes one of `teacher`, `collect`, `train`, or `eval` to `/content/monopoly_stage.txt`. Every expensive stage resumes only when its code/config fingerprint matches restored artifacts. After each stage the launcher downloads an atomic run snapshot to the laptop. The generic `Gemma4_12B_15GB_Colab_QLoRA_Test.ipynb` remains the hardware regression reference.


In [ ]:
# Stage selection, dependency setup, and secret-free project extraction.
from __future__ import annotations

import gc
import importlib.util
import os
import re
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

STAGE_FILE = Path("/content/monopoly_stage.txt")
RUN_STAGE = STAGE_FILE.read_text(encoding="utf-8").strip() if STAGE_FILE.exists() else "teacher"
if RUN_STAGE not in {"teacher", "collect", "train", "eval", "all"}:
    raise ValueError(f"Unknown stage: {RUN_STAGE}")
print("Requested stage:", RUN_STAGE)

if importlib.util.find_spec("unsloth") is None:
    import torch
    version_match = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__))
    torch_minor = version_match.group(0) if version_match else "2.10"
    xformers = "xformers==" + {
        "2.10": "0.0.34",
        "2.9": "0.0.33.post1",
        "2.8": "0.0.32.post2",
    }.get(torch_minor, "0.0.34")
    commands = [
        [sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "protobuf",
         "datasets==4.3.0", "huggingface_hub>=0.34.0", "hf_transfer", "psutil"],
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "unsloth_zoo",
         "bitsandbytes", "accelerate", xformers, "peft", "trl", "triton", "unsloth"],
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade", "torchao>=0.16.0"],
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
         "transformers==5.10.1", "tokenizers>=0.22.0,<=0.23.0"],
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=1.5.0,<2.0"],
    ]
    for command in commands:
        subprocess.run(command, check=True)

ARCHIVE = Path("/content/DeepRL_Monopoly.tar.gz")
PROJECT_ROOT = Path("/content/DeepRL_Monopoly")
if not ARCHIVE.exists():
    raise FileNotFoundError("The launcher must upload /content/DeepRL_Monopoly.tar.gz")
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
with tarfile.open(ARCHIVE, "r:gz") as archive:
    archive.extractall("/content", filter="data")
if (PROJECT_ROOT / ".env").exists():
    raise RuntimeError("Secret-safety failure: .env was present in the uploaded archive")

PPO_ROOT = PROJECT_ROOT / "RL_PPO(UNOFFICIAL)_MONOPOLY"
SLM_ROOT = PROJECT_ROOT / "SLM_HANDMADE_MONOPOLY"
sys.path[:0] = [str(PPO_ROOT), str(SLM_ROOT), str(PROJECT_ROOT)]

try:
    from google.colab import userdata
except Exception:
    userdata = None
try:
    HF_TOKEN = userdata.get("HF_TOKEN") if userdata is not None else None
except Exception:
    HF_TOKEN = None
print("HF_TOKEN loaded from Colab Secrets." if HF_TOKEN else "No HF_TOKEN secret; using public access.")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA T4 runtime is required.")
gpu = torch.cuda.get_device_properties(0)
total_vram_gib = gpu.total_memory / 1024**3
if "T4" not in gpu.name.upper() or total_vram_gib < 13.5:
    raise RuntimeError(
        f"Expected a T4 with >=13.5 GiB usable VRAM; found {gpu.name} ({total_vram_gib:.2f} GiB)."
    )
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True, check=False,
).stdout.strip())
PREFLIGHT_OK = True


In [ ]:
# Versioned run root, fingerprints, atomic writes, and stage resource guards.
if not globals().get("PREFLIGHT_OK"):
    raise RuntimeError("Preflight did not complete; refusing to create artifacts")
import hashlib
import importlib.metadata
import json
import math
import signal
import time
from datetime import datetime, timezone

import psutil

from monopoly_qlora import (
    SCHEMA_VERSION,
    canonical_json,
    file_sha256,
    sha256_text,
)

RUN_ROOT = Path("/content/pilot_v1")
TEACHER_DIR = RUN_ROOT / "teacher"
DATASET_DIR = RUN_ROOT / "datasets"
ADAPTER_DIR = RUN_ROOT / "adapters"
EVAL_DIR = RUN_ROOT / "eval"
NOTEBOOK_DIR = RUN_ROOT / "notebook_output"
for directory in (RUN_ROOT, TEACHER_DIR, DATASET_DIR, ADAPTER_DIR, EVAL_DIR, NOTEBOOK_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = RUN_ROOT / "manifest.json"
CONFIG = {
    "schema": SCHEMA_VERSION,
    "ruleset": "ppo-plus-v2",
    "model": "unsloth/gemma-4-12b-it",
    "max_seq_length": 512,
    "teacher_initial_games": 2000,
    "teacher_increment_games": 1000,
    "teacher_max_games": 10000,
    "teacher_eval_games": 600,
    "teacher_min_win_rate": 0.35,
    "teacher_min_wilson_low": 0.25,
    "train_rows": 2048,
    "validation_rows": 256,
    "test_rows": 256,
    "rollouts_per_action": 4,
    "rollout_horizon": 256,
    "candidate_limit": 16,
    "relabel_margin": 0.05,
    "min_loss_fraction": 0.25,
    "lora_r": 4,
    "lora_alpha": 4,
    "micro_batch": 1,
    "gradient_accumulation": 8,
    "learning_rate": 2e-4,
    "epochs": 1,
    "eval_save_steps": 64,
    "seed": 3407,
}
CODE_HASH = file_sha256(SLM_ROOT / "monopoly_qlora.py")
CONFIG_HASH = sha256_text(canonical_json(CONFIG))

def atomic_json(path: Path, value) -> None:
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=True, indent=2, sort_keys=True, default=str)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def atomic_text(path: Path, value: str) -> None:
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        handle.write(value)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def atomic_jsonl(path: Path, rows) -> None:
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=True, separators=(",", ":"), default=str) + "\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def read_json(path: Path, default):
    if not path.exists():
        return default
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

manifest = read_json(MANIFEST_PATH, {
    "pilot": "pilot_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "stages": {},
})
manifest.update(config=CONFIG, config_hash=CONFIG_HASH, code_hash=CODE_HASH)
atomic_json(MANIFEST_PATH, manifest)

def stage_fingerprint(name: str, extra=None) -> str:
    return sha256_text(canonical_json({
        "stage": name,
        "config": CONFIG,
        "code_hash": CODE_HASH,
        "extra": extra or {},
    }))

def stage_done(name: str, fingerprint: str, required=()) -> bool:
    state = manifest.get("stages", {}).get(name, {})
    return (
        state.get("status") == "complete"
        and state.get("fingerprint") == fingerprint
        and all(Path(path).exists() for path in required)
    )

def mark_stage(name: str, fingerprint: str, status: str, **details) -> None:
    manifest.setdefault("stages", {})[name] = {
        "status": status,
        "fingerprint": fingerprint,
        "updated_at": datetime.now(timezone.utc).isoformat(),
        **details,
    }
    atomic_json(MANIFEST_PATH, manifest)

class StageGuard:
    def __init__(
        self,
        hours: float,
        min_available_ram_gib: float = 2.0,
        min_disk_gib: float = 10.0,
        min_free_vram_gib: float = 0.25,
    ):
        self.deadline = time.monotonic() + hours * 3600
        self.min_available_ram = min_available_ram_gib * 1024**3
        self.min_disk = min_disk_gib * 1024**3
        self.min_free_vram = min_free_vram_gib * 1024**3

    def check(self):
        if time.monotonic() >= self.deadline:
            raise RuntimeError("Stage hard deadline reached")
        available = psutil.virtual_memory().available
        if available < self.min_available_ram:
            raise RuntimeError(f"Available RAM below guard: {available / 1024**3:.2f} GiB")
        for root in ("/content", RUN_ROOT):
            free = shutil.disk_usage(root).free
            if free < self.min_disk:
                raise RuntimeError(f"Free disk below guard at {root}: {free / 1024**3:.2f} GiB")
        free_vram, _ = torch.cuda.mem_get_info()
        if free_vram < self.min_free_vram:
            raise RuntimeError(f"Free VRAM below guard: {free_vram / 1024**3:.2f} GiB")
        return True

versions = {
    name: importlib.metadata.version(name)
    for name in ("torch", "transformers", "trl", "unsloth", "datasets", "peft", "bitsandbytes")
    if importlib.util.find_spec(name) is not None
}
preflight = {
    "stage": RUN_STAGE,
    "gpu": gpu.name,
    "total_vram_gib": total_vram_gib,
    "python": sys.version,
    "versions": versions,
    "config_hash": CONFIG_HASH,
    "code_hash": CODE_HASH,
    "timestamp": datetime.now(timezone.utc).isoformat(),
}
atomic_json(NOTEBOOK_DIR / f"preflight_{RUN_STAGE}.json", preflight)
print(json.dumps(preflight, indent=2))


## Stage 1 — gated PPO teacher

The hybrid PPO teacher is trained in seat 0 against randomized three-of-six scripted groups. Evaluation is deterministic over 600 disjoint seeded games with rotating seats. A failed 35%/Wilson gate stops the pipeline before any QLoRA labels are accepted.


In [ ]:
def run_teacher_stage():
    if not globals().get("PREFLIGHT_OK"):
        raise RuntimeError("Preflight did not complete; teacher training is disabled")
    from monopoly_drl.agent_ppo import PPOAgent
    from monopoly_qlora import evaluate_teacher, train_teacher_block

    fingerprint = stage_fingerprint("teacher")
    checkpoint = TEACHER_DIR / "ppo_plus_v2_teacher.pt"
    gate_path = TEACHER_DIR / "teacher_gate.json"
    if stage_done("teacher", fingerprint, (checkpoint, gate_path)):
        print("Teacher stage already complete with matching fingerprint.")
        return

    resume_meta_path = TEACHER_DIR / "resume_fingerprint.json"
    resume_meta = read_json(resume_meta_path, {})
    if checkpoint.exists() and resume_meta.get("fingerprint") != fingerprint:
        raise RuntimeError("Teacher checkpoint fingerprint mismatch; use a new pilot root")
    atomic_json(resume_meta_path, {"fingerprint": fingerprint})
    guard = StageGuard(hours=12, min_disk_gib=12)
    agent = PPOAgent(player_id=0, hybrid=True, device="cuda")
    if checkpoint.exists():
        agent.load(str(checkpoint))
        print("Resumed teacher from", agent.games_trained, "games")

    histories = read_json(TEACHER_DIR / "training_history.json", [])
    gate = read_json(gate_path, None)
    while True:
        gate_is_current = (
            checkpoint.exists()
            and isinstance(gate, dict)
            and gate.get("games_trained") == agent.games_trained
        )
        if gate_is_current and gate.get("passed"):
            gate.setdefault("checkpoint_hash", file_sha256(checkpoint))
            mark_stage(
                "teacher",
                fingerprint,
                "complete",
                checkpoint=str(checkpoint),
                checkpoint_hash=gate["checkpoint_hash"],
                games_trained=agent.games_trained,
                gate=gate,
            )
            return

        needs_evaluation = (
            agent.games_trained >= CONFIG["teacher_initial_games"]
            and not gate_is_current
        )
        if not needs_evaluation:
            if agent.games_trained >= CONFIG["teacher_max_games"]:
                break
            target = (
                CONFIG["teacher_initial_games"]
                if agent.games_trained < CONFIG["teacher_initial_games"]
                else min(
                    agent.games_trained + CONFIG["teacher_increment_games"],
                    CONFIG["teacher_max_games"],
                )
            )
            games = target - agent.games_trained
            try:
                histories.extend(train_teacher_block(
                    agent,
                    games,
                    seed=CONFIG["seed"],
                    checkpoint_path=checkpoint,
                    checkpoint_every=100,
                    watchdog=guard,
                ))
            except Exception:
                agent.save(str(checkpoint))
                atomic_json(TEACHER_DIR / "training_history.json", histories)
                raise
            atomic_json(TEACHER_DIR / "training_history.json", histories)

        gate = evaluate_teacher(
            agent,
            games=CONFIG["teacher_eval_games"],
            seed=90_000,
            watchdog=guard,
        )
        gate["games_trained"] = agent.games_trained
        gate["checkpoint_hash"] = file_sha256(checkpoint)
        atomic_json(gate_path, gate)
        print({
            "games_trained": agent.games_trained,
            "win_rate": gate["win_rate"],
            "wilson_low": gate["wilson_95"][0],
            "passed": gate["passed"],
        })
        if gate["passed"]:
            mark_stage(
                "teacher",
                fingerprint,
                "complete",
                checkpoint=str(checkpoint),
                checkpoint_hash=gate["checkpoint_hash"],
                games_trained=agent.games_trained,
                gate=gate,
            )
            return
        if agent.games_trained >= CONFIG["teacher_max_games"]:
            break

    failure = {
        "reason": "Teacher failed the pilot strength gate",
        "gate": gate,
        "config_hash": CONFIG_HASH,
        "code_hash": CODE_HASH,
    }
    atomic_json(RUN_ROOT / "TEACHER_GATE_FAILED.json", failure)
    mark_stage("teacher", fingerprint, "failed", failure=failure)
    raise RuntimeError("TEACHER_GATE_FAILED: stopping before collection and QLoRA")

if RUN_STAGE in {"teacher", "all"}:
    run_teacher_stage()


## Stage 2 — collection and rollout relabeling

Only non-forced PPO decisions are collected. At most 16 candidates receive four common-random-number rollouts to a 256-decision horizon. Low-margin labels are discarded, exact prompt states are deduplicated, loss-game states must be at least 25%, and game IDs never cross splits. Rows are response-tokenized without truncation before atomic saves.


In [ ]:
def run_collection_stage():
    if not globals().get("PREFLIGHT_OK"):
        raise RuntimeError("Preflight did not complete; collection is disabled")
    from transformers import AutoTokenizer

    from monopoly_drl.agent_ppo import PPOAgent
    from monopoly_drl.env import MonopolyEnv
    from monopoly_qlora import (
        collect_relabelled_game,
        scripted_opponents,
        split_by_game,
        tokenize_rows,
        validate_splits,
    )

    teacher_checkpoint = TEACHER_DIR / "ppo_plus_v2_teacher.pt"
    teacher_gate = read_json(TEACHER_DIR / "teacher_gate.json", {})
    if not teacher_checkpoint.exists() or not teacher_gate.get("passed"):
        raise RuntimeError("A passing PPO teacher checkpoint is required before collection")
    teacher_hash = file_sha256(teacher_checkpoint)
    fingerprint = stage_fingerprint("collect", {"teacher_hash": teacher_hash})
    split_paths = {
        "train": DATASET_DIR / "train.jsonl",
        "validation": DATASET_DIR / "validation.jsonl",
        "test": DATASET_DIR / "test.jsonl",
    }
    if stage_done("collect", fingerprint, split_paths.values()):
        print("Collection stage already complete with matching fingerprint.")
        return

    guard = StageGuard(hours=18, min_disk_gib=12)
    teacher = PPOAgent(player_id=0, hybrid=True, device="cuda")
    teacher.load(str(teacher_checkpoint))

    progress_path = DATASET_DIR / "collection_progress.json"
    progress = read_json(progress_path, {"fingerprint": fingerprint, "next_seed": 200_000, "rows": [], "games": []})
    if progress.get("fingerprint") != fingerprint:
        raise RuntimeError("Collection checkpoint fingerprint mismatch; use a new pilot root")
    rows = progress["rows"]
    sizes = {
        "train": CONFIG["train_rows"],
        "validation": CONFIG["validation_rows"],
        "test": CONFIG["test_rows"],
    }
    splits = None

    while splits is None:
        guard.check()
        if len(rows) >= sum(sizes.values()):
            try:
                candidate = split_by_game(rows, sizes, seed=CONFIG["seed"])
                validate_splits(candidate)
                loss_fractions = {
                    name: sum(row["outcome"] == "loss" for row in values) / len(values)
                    for name, values in candidate.items()
                }
                if min(loss_fractions.values()) >= CONFIG["min_loss_fraction"]:
                    splits = candidate
                    print("Split loss fractions:", loss_fractions)
                    break
            except ValueError:
                pass

        seed = int(progress["next_seed"])
        teacher_pid = seed % 4
        env = MonopolyEnv(agent_ids=[teacher_pid], max_rounds=200)
        opponents = scripted_opponents(seed, teacher_pid)
        game_rows, game_report = collect_relabelled_game(
            env=env,
            teacher=teacher,
            teacher_pid=teacher_pid,
            opponents=opponents,
            game_id=f"collect-{seed}",
            seed=seed,
            teacher_checkpoint_hash=teacher_hash,
            rollouts_per_action=CONFIG["rollouts_per_action"],
            rollout_horizon=CONFIG["rollout_horizon"],
            candidate_limit=CONFIG["candidate_limit"],
            min_margin=CONFIG["relabel_margin"],
            watchdog=guard,
        )
        rows.extend(game_rows)
        progress["games"].append(game_report)
        progress["next_seed"] = seed + 1
        progress["rows"] = rows
        atomic_json(progress_path, progress)
        print("Collected", len(rows), "eligible rows through seed", seed)

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"], token=HF_TOKEN)
    dataset_hashes = {}
    for name, split_rows in splits.items():
        tokenized = tokenize_rows(split_rows, tokenizer, CONFIG["max_seq_length"])
        atomic_jsonl(split_paths[name], tokenized)
        dataset_hashes[name] = file_sha256(split_paths[name])

    metadata = {
        "sizes": {name: len(values) for name, values in splits.items()},
        "loss_fractions": {
            name: sum(row["outcome"] == "loss" for row in values) / len(values)
            for name, values in splits.items()
        },
        "dataset_hashes": dataset_hashes,
        "teacher_checkpoint_hash": teacher_hash,
        "config_hash": CONFIG_HASH,
        "code_hash": CODE_HASH,
        "max_token_count": max(
            json.loads(line)["token_count"]
            for path in split_paths.values()
            for line in path.read_text(encoding="utf-8").splitlines()
        ),
    }
    atomic_json(DATASET_DIR / "metadata.json", metadata)
    mark_stage("collect", fingerprint, "complete", **metadata)
    print(json.dumps(metadata, indent=2))

if RUN_STAGE in {"collect", "all"}:
    run_collection_stage()


## Stage 3 — Gemma 4 12B QLoRA SFT

This stage uses the proven T4 profile: 512 tokens, 4-bit text-only base, attention-only LoRA rank/alpha 4, micro-batch 1, accumulation 8, Unsloth gradient checkpointing, and response-only labels prepared in the collection stage. It runs one epoch, evaluating and checkpointing every 64 optimizer steps, and resumes the latest restored checkpoint.


In [ ]:
def run_training_stage():
    if not globals().get("PREFLIGHT_OK"):
        raise RuntimeError("Preflight did not complete; QLoRA is disabled")
    import zipfile

    from datasets import Dataset
    from transformers import TrainerCallback, default_data_collator
    from transformers.trainer_utils import get_last_checkpoint
    from trl import SFTConfig, SFTTrainer
    from unsloth import FastModel, is_bfloat16_supported

    train_path = DATASET_DIR / "train.jsonl"
    validation_path = DATASET_DIR / "validation.jsonl"
    if not train_path.exists() or not validation_path.exists():
        raise RuntimeError("Tokenized train and validation datasets are required")
    dataset_hashes = {
        "train": file_sha256(train_path),
        "validation": file_sha256(validation_path),
    }
    fingerprint = stage_fingerprint("train", dataset_hashes)
    final_adapter = ADAPTER_DIR / "gemma4_12b_monopoly_lora"
    archive_path = ADAPTER_DIR / "gemma4_12b_monopoly_lora.zip"
    samples_path = ADAPTER_DIR / "sample_generations.json"
    if stage_done("train", fingerprint, (final_adapter / "adapter_config.json", archive_path, samples_path)):
        print("QLoRA stage already complete with matching fingerprint.")
        return

    guard = StageGuard(hours=10, min_disk_gib=15, min_free_vram_gib=0.15)
    guard.check()
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model, tokenizer = FastModel.from_pretrained(
        model_name=CONFIG["model"],
        max_seq_length=CONFIG["max_seq_length"],
        dtype=None,
        load_in_4bit=True,
        full_finetuning=False,
        text_only=True,
        offload_embedding=True,
        token=HF_TOKEN,
    )
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_audio_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=False,
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=CONFIG["seed"],
    )

    def load_tokens(path):
        rows = [
            json.loads(line)
            for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        return Dataset.from_list([
            {key: row[key] for key in ("input_ids", "attention_mask", "labels")}
            for row in rows
        ])

    train_dataset = load_tokens(train_path)
    validation_dataset = load_tokens(validation_path)
    checkpoint_dir = ADAPTER_DIR / "checkpoints"
    checkpoint_fingerprint = ADAPTER_DIR / "checkpoint_fingerprint.json"
    previous_fingerprint = read_json(checkpoint_fingerprint, {})
    if any(checkpoint_dir.glob("checkpoint-*")) and previous_fingerprint.get("fingerprint") != fingerprint:
        raise RuntimeError("QLoRA checkpoint fingerprint mismatch; use a new pilot root")
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    atomic_json(checkpoint_fingerprint, {"fingerprint": fingerprint})

    class GuardCallback(TrainerCallback):
        def on_step_end(self, args, state, control, **kwargs):
            guard.check()
            return control

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        data_collator=default_data_collator,
        callbacks=[GuardCallback()],
        args=SFTConfig(
            output_dir=str(checkpoint_dir),
            max_length=CONFIG["max_seq_length"],
            dataset_kwargs={"skip_prepare_dataset": True},
            packing=False,
            padding_free=False,
            per_device_train_batch_size=CONFIG["micro_batch"],
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=CONFIG["gradient_accumulation"],
            num_train_epochs=CONFIG["epochs"],
            learning_rate=CONFIG["learning_rate"],
            warmup_ratio=0.03,
            logging_steps=1,
            eval_strategy="steps",
            eval_steps=CONFIG["eval_save_steps"],
            save_strategy="steps",
            save_steps=CONFIG["eval_save_steps"],
            save_total_limit=4,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=CONFIG["seed"],
            report_to="none",
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            remove_unused_columns=False,
        ),
    )

    import torch.nn.functional as F

    if not hasattr(F, "_gemma4_original_sdpa"):
        F._gemma4_original_sdpa = F.scaled_dot_product_attention

    def _gemma4_safe_sdpa(query, key, value, *args, **kwargs):
        target_dtype = query.dtype
        if key.dtype != target_dtype:
            key = key.to(target_dtype)
        if value.dtype != target_dtype:
            value = value.to(target_dtype)
        return F._gemma4_original_sdpa(query, key, value, *args, **kwargs)

    F.scaled_dot_product_attention = _gemma4_safe_sdpa
    latest = get_last_checkpoint(str(checkpoint_dir))
    try:
        result = trainer.train(resume_from_checkpoint=latest)
    except Exception:
        trainer.save_state()
        raise

    temporary_adapter = final_adapter.with_name(final_adapter.name + ".tmp")
    if temporary_adapter.exists():
        shutil.rmtree(temporary_adapter)
    model.save_pretrained(temporary_adapter)
    tokenizer.save_pretrained(temporary_adapter)
    if final_adapter.exists():
        shutil.rmtree(final_adapter)
    os.replace(temporary_adapter, final_adapter)
    trainer.save_state()
    trainer.save_metrics("train", result.metrics)
    shutil.make_archive(str(archive_path.with_suffix("")), "zip", final_adapter)

    FastModel.for_inference(model)
    validation_rows = [json.loads(line) for line in validation_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    samples = []
    for row in validation_rows[:8]:
        guard.check()
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": row["prompt"]}],
            tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
        ).to("cuda")
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=64, do_sample=False, use_cache=True)
        raw = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        samples.append({"state_hash": row["state_hash"], "expected": row["completion"], "raw_output": raw})
    atomic_json(samples_path, samples)

    lock = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
    atomic_text(ADAPTER_DIR / "package-lock.txt", lock)
    artifacts = {
        "adapter_hash": file_sha256(final_adapter / "adapter_model.safetensors"),
        "archive_hash": file_sha256(archive_path),
        "sample_generations_hash": file_sha256(samples_path),
        "dataset_hashes": dataset_hashes,
        "config_hash": CONFIG_HASH,
        "code_hash": CODE_HASH,
        "metrics": result.metrics,
        "peak_vram_gib": torch.cuda.max_memory_reserved() / 1024**3,
    }
    atomic_json(ADAPTER_DIR / "training_metrics.json", artifacts)
    mark_stage("train", fingerprint, "complete", **artifacts)
    print(json.dumps(artifacts, indent=2, default=str))

if RUN_STAGE in {"train", "all"}:
    run_training_stage()


## Stage 4 — offline and standalone simulator evaluation

The adapter must first clear 98% parseable JSON, 97% legal-without-fallback, and 65% rollout-label agreement on the held-out split. Then eight smoke games and 32 paired-seed Gemma/PPO games run in randomized seats. The largest inference batch in 1/2/4/8 that preserves at least 1 GiB free VRAM is selected. Failed, invalid, and high-regret prompts are retained for DPO.


In [ ]:
def run_evaluation_stage():
    if not globals().get("PREFLIGHT_OK"):
        raise RuntimeError("Preflight did not complete; evaluation is disabled")
    from unsloth import FastModel

    from monopoly_drl.agent_ppo import PPOAgent
    from monopoly_drl.env import MonopolyEnv
    from monopoly_qlora import (
        deterministic_ppo_action,
        parse_action_json,
        play_model_game,
        play_policy_game,
        scripted_opponents,
    )

    adapter = ADAPTER_DIR / "gemma4_12b_monopoly_lora"
    test_path = DATASET_DIR / "test.jsonl"
    teacher_checkpoint = TEACHER_DIR / "ppo_plus_v2_teacher.pt"
    if not (adapter / "adapter_config.json").exists() or not test_path.exists():
        raise RuntimeError("Adapter and held-out dataset are required")
    fingerprint = stage_fingerprint("eval", {
        "adapter": file_sha256(adapter / "adapter_model.safetensors"),
        "test": file_sha256(test_path),
        "teacher": file_sha256(teacher_checkpoint),
    })
    final_report_path = EVAL_DIR / "final_report.json"
    if stage_done("eval", fingerprint, (final_report_path,)):
        print("Evaluation stage already complete with matching fingerprint.")
        return

    guard = StageGuard(hours=18, min_disk_gib=12, min_free_vram_gib=0.15)
    model, tokenizer = FastModel.from_pretrained(
        model_name=str(adapter),
        max_seq_length=CONFIG["max_seq_length"],
        dtype=None,
        load_in_4bit=True,
        token=HF_TOKEN,
    )
    FastModel.for_inference(model)
    tokenizer.padding_side = "left"

    def generate_batch(prompts):
        guard.check()
        conversations = [[{"role": "user", "content": prompt}] for prompt in prompts]
        inputs = tokenizer.apply_chat_template(
            conversations,
            tokenize=True,
            add_generation_prompt=True,
            padding=True,
            return_tensors="pt",
            return_dict=True,
        ).to("cuda")
        remaining = CONFIG["max_seq_length"] - inputs["input_ids"].shape[1]
        if remaining <= 0:
            raise RuntimeError("Serialized state exceeds the model context; truncation is forbidden")
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=min(64, remaining),
                do_sample=False,
                use_cache=True,
            )
        generated = outputs[:, inputs["input_ids"].shape[1]:]
        return [text.strip() for text in tokenizer.batch_decode(generated, skip_special_tokens=True)]

    test_rows = [
        json.loads(line)
        for line in test_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    benchmark = []
    selected_batch = None
    for batch_size in (1, 2, 4, 8):
        try:
            gc.collect()
            torch.cuda.empty_cache()
            started = time.perf_counter()
            prompts = [test_rows[index % len(test_rows)]["prompt"] for index in range(batch_size)]
            generate_batch(prompts)
            elapsed = time.perf_counter() - started
            free_gib = torch.cuda.mem_get_info()[0] / 1024**3
            benchmark.append({
                "batch_size": batch_size,
                "latency_s": elapsed,
                "free_vram_gib": free_gib,
                "ok": free_gib >= 1.0,
            })
            if free_gib >= 1.0:
                selected_batch = batch_size
        except torch.cuda.OutOfMemoryError as exc:
            benchmark.append({"batch_size": batch_size, "ok": False, "error": str(exc)})
            gc.collect()
            torch.cuda.empty_cache()
    if selected_batch is None:
        raise RuntimeError("No inference batch size retains the required 1 GiB VRAM headroom")
    atomic_json(EVAL_DIR / "batch_benchmark.json", benchmark)

    raw_outputs = []
    for start in range(0, len(test_rows), selected_batch):
        raw_outputs.extend(generate_batch([
            row["prompt"] for row in test_rows[start:start + selected_batch]
        ]))

    parseable = legal = exact = 0
    failures = []
    high_regret = []
    for row, raw in zip(test_rows, raw_outputs):
        try:
            parsed_object = json.loads(raw)
            if isinstance(parsed_object, dict):
                parseable += 1
        except json.JSONDecodeError:
            parsed_object = None
        validation_env = MonopolyEnv(agent_ids=[row["actor_pid"]], max_rounds=1)
        validation_env.turn_order = list(row["seat_order"])
        try:
            action = parse_action_json(
                raw,
                validation_env,
                row["actor_pid"],
                row["legal_actions"],
            )
            legal += 1
            if action == row["relabeled_action"]:
                exact += 1
            else:
                scores = row["rollout_scores"]
                if str(action) in scores:
                    best = np.mean(scores[str(row["relabeled_action"])])
                    predicted = np.mean(scores[str(action)])
                    high_regret.append({
                        "state_hash": row["state_hash"],
                        "prompt": row["prompt"],
                        "raw_output": raw,
                        "predicted_action": action,
                        "expected_action": row["relabeled_action"],
                        "regret": float(best - predicted),
                        "rollout_scores": scores,
                    })
        except Exception as exc:
            failures.append({
                "state_hash": row["state_hash"],
                "prompt": row["prompt"],
                "raw_output": raw,
                "expected": row["completion"],
                "error": str(exc),
            })

    offline = {
        "rows": len(test_rows),
        "parseable_rate": parseable / len(test_rows),
        "legal_rate": legal / len(test_rows),
        "exact_rate": exact / len(test_rows),
        "selected_batch_size": selected_batch,
    }
    offline["passed"] = (
        offline["parseable_rate"] >= 0.98
        and offline["legal_rate"] >= 0.97
        and offline["exact_rate"] >= 0.65
    )
    atomic_json(EVAL_DIR / "offline_metrics.json", offline)
    atomic_jsonl(EVAL_DIR / "invalid_states.jsonl", failures)
    atomic_jsonl(EVAL_DIR / "high_regret_states.jsonl", high_regret)

    if not offline["passed"]:
        report = {"offline": offline, "promoted": False, "reason": "offline gates failed"}
        atomic_json(final_report_path, report)
        mark_stage("eval", fingerprint, "complete", **report)
        print(json.dumps(report, indent=2))
        return

    def generate_one(prompt):
        return generate_batch([prompt])[0]

    smoke_reports = []
    for index in range(8):
        seed = 300_000 + index
        seat = index % 4
        random.seed(seed)
        np.random.seed(seed)
        try:
            game = play_model_game(
                MonopolyEnv(agent_ids=[seat], max_rounds=200),
                seat,
                generate_one,
                scripted_opponents(seed, seat),
            )
        except Exception as exc:
            game = {"finished": False, "model_won": False, "decisions": [], "crash": str(exc)}
        smoke_reports.append({"seed": seed, "seat": seat, **game})
    atomic_json(EVAL_DIR / "smoke_games.json", smoke_reports)
    smoke_passed = all(report["finished"] for report in smoke_reports)

    teacher = PPOAgent(player_id=0, hybrid=True, device="cuda")
    teacher.load(str(teacher_checkpoint))
    paired = []
    gemma_wins = teacher_wins = 0
    for index in range(32):
        seed = 310_000 + index
        seat = index % 4

        random.seed(seed)
        np.random.seed(seed)
        try:
            gemma = play_model_game(
                MonopolyEnv(agent_ids=[seat], max_rounds=200),
                seat,
                generate_one,
                scripted_opponents(seed, seat),
            )
        except Exception as exc:
            gemma = {"finished": False, "model_won": False, "decisions": [], "crash": str(exc)}
        gemma_wins += gemma["model_won"]

        random.seed(seed)
        np.random.seed(seed)
        teacher_game = play_policy_game(
            MonopolyEnv(agent_ids=[seat], max_rounds=200),
            seat,
            lambda state, pid: deterministic_ppo_action(teacher, state, pid)[0],
            scripted_opponents(seed, seat),
        )
        teacher_wins += teacher_game["winner"] == seat
        paired.append({
            "seed": seed,
            "seat": seat,
            "gemma": gemma,
            "teacher": teacher_game,
        })
    atomic_json(EVAL_DIR / "paired_games.json", paired)

    game_metrics = {
        "smoke_passed": smoke_passed,
        "gemma_wins": gemma_wins,
        "teacher_wins": teacher_wins,
        "games": 32,
        "gemma_win_rate": gemma_wins / 32,
        "teacher_win_rate": teacher_wins / 32,
        "paired_finished": all(pair["gemma"].get("finished", False) and pair["teacher"].get("finished", False) for pair in paired),
    }
    game_metrics["passed"] = (
        smoke_passed
        and game_metrics["paired_finished"]
        and game_metrics["gemma_win_rate"] >= 0.25
        and game_metrics["gemma_win_rate"] >= game_metrics["teacher_win_rate"] - 0.10
    )
    game_invalid = [
        decision
        for report in smoke_reports
        for decision in report["decisions"]
        if decision["fallback"]
    ] + [
        decision
        for pair in paired
        for decision in pair["gemma"]["decisions"]
        if decision["fallback"]
    ]
    atomic_jsonl(EVAL_DIR / "game_invalid_states.jsonl", game_invalid)
    game_crashes = [report for report in smoke_reports if report.get("crash")] + [pair["gemma"] for pair in paired if pair["gemma"].get("crash")]
    atomic_jsonl(EVAL_DIR / "game_crashes.jsonl", game_crashes)

    report = {
        "offline": offline,
        "games": game_metrics,
        "benchmark": benchmark,
        "promoted": bool(offline["passed"] and game_metrics["passed"]),
        "adapter_retained": True,
    }
    atomic_json(final_report_path, report)
    mark_stage("eval", fingerprint, "complete", **report)
    print(json.dumps(report, indent=2))

if RUN_STAGE in {"eval", "all"}:
    run_evaluation_stage()


In [ ]:
# Final manifest snapshot for the requested stage.
manifest = read_json(MANIFEST_PATH, manifest)
print(json.dumps({
    "run_root": str(RUN_ROOT),
    "requested_stage": RUN_STAGE,
    "stages": manifest.get("stages", {}),
}, indent=2, default=str))
